# 电商订单处理多智能体系统

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv()  # 读取当前目录下的 .env

model_name = os.getenv("OPENAI_MODE_NAME", "gpt-4o")
model = init_chat_model(model_name)
print("模型初始化完成，模型：", model_name)

模型初始化完成，模型： gpt-5.4-nano


## 第 1 步：模拟业务数据

In [4]:
ORDERS = {
    "ORD-20240115": {
        "status": "shipped",
        "items": [{"sku": "SKU-EARPHONE-01", "name": "无线耳机", "qty": 1}],
        "address": "北京市海淀区中关村大街 1 号",
        "order_time": "2024-01-15 10:23:00",
    },
    "ORD-20240116": {
        "status": "pending",
        "items": [{"sku": "SKU-CHARGER-02", "name": "充电器", "qty": 2}],
        "address": "上海市浦东新区世纪大道 100 号",
        "order_time": "2024-01-16 09:10:00",
    },
}

INVENTORY = {
    ("SKU-EARPHONE-01", "WH-BJ"): {"current_stock": 15, "safety_threshold": 20, "in_transit": 50},
    ("SKU-CHARGER-02", "WH-BJ"): {"current_stock": 8, "safety_threshold": 30, "in_transit": 0},
}

SHIPMENTS = {
    "ORD-20240115": {
        "carrier": "顺丰",
        "tracking_no": "SF1234567890",
        "current_location": "北京市朝阳区转运中心",
        "eta": "2024-01-16",
    },
}
print("模拟数据已就绪")

模拟数据已就绪


## 第 2 步：定义 6 个领域工具

和 `07` 里的 `create_calendar_event` / `send_email` 完全同一写法：`@tool` + 类型注解 + docstring。

In [5]:
from langchain.tools import tool

@tool
def query_order_status(order_id: str) -> dict:
    """查询订单详情：状态、商品、地址、时间等。"""
    order = ORDERS.get(order_id)
    if order is None:
        return {"error": f"订单 {order_id} 不存在"}
    return {"status": order["status"], "items": order["items"], "address": order["address"], "order_time": order["order_time"]}

@tool
def update_order_status(order_id: str, new_status: str, reason: str) -> str:
    """申请变更订单状态（如取消、退款、完成）。"""
    return f"订单 {order_id} 状态变更申请已提交，待审核"

@tool
def check_inventory(sku: str, warehouse: str) -> dict:
    """查询商品库存：当前数量、安全阈值、在途量。"""
    data = INVENTORY.get((sku, warehouse))
    if data is None:
        return {"error": f"{sku} 在 {warehouse} 无库存记录"}
    return {"sku": sku, "warehouse": warehouse, "current_stock": data["current_stock"],
            "safety_threshold": data["safety_threshold"], "in_transit": data["in_transit"]}

@tool
def adjust_inventory_threshold(sku: str, new_threshold: int) -> str:
    """申请调整商品安全库存阈值。"""
    return f"{sku} 阈值调整申请已提交，待审核"

@tool
def suggest_replenish(sku: str, suggested_qty: int) -> str:
    """生成补货建议单。"""
    return f"补货建议单已生成：{sku} 建议补货 {suggested_qty} 件，待采购确认"

@tool
def track_shipment(order_id: str) -> dict:
    """跟踪已发货订单的物流轨迹与时效。"""
    shipment = SHIPMENTS.get(order_id)
    if shipment is None:
        return {"error": f"订单 {order_id} 暂无物流信息"}
    return {"carrier": shipment["carrier"], "tracking_no": shipment["tracking_no"],
            "current_location": shipment["current_location"], "eta": shipment["eta"]}

print("6 个领域工具已定义")

6 个领域工具已定义


## 第 3 步：创建三个子智能体

In [6]:
ORDER_PROMPT = """
你是一个订单智能体，负责查询订单状态、申请变更订单状态。
可用工具：
- query_order_status(order_id)
- update_order_status(order_id, new_status, reason)
请用中文回答。
"""
order_agent = create_agent(model=model, tools=[query_order_status, update_order_status], system_prompt=ORDER_PROMPT)

INVENTORY_PROMPT = """
你是一个库存智能体，负责查询库存、申请调整安全阈值、生成补货建议。
仓库标识默认使用 WH-BJ。
可用工具：
- check_inventory(sku, warehouse)
- adjust_inventory_threshold(sku, new_threshold)
- suggest_replenish(sku, suggested_qty)
请用中文回答。
"""
inventory_agent = create_agent(model=model, tools=[check_inventory, adjust_inventory_threshold, suggest_replenish], system_prompt=INVENTORY_PROMPT)

SHIPPING_PROMPT = """
你是一个物流智能体，负责跟踪已发货订单的物流轨迹与时效。
可用工具：
- track_shipment(order_id)
请用中文回答。
"""
shipping_agent = create_agent(model=model, tools=[track_shipment], system_prompt=SHIPPING_PROMPT)

print("三个子智能体已创建：order_agent / inventory_agent / shipping_agent")

三个子智能体已创建：order_agent / inventory_agent / shipping_agent


## 第 4 步：把子智能体包装成工具 + 创建主管智能体

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

@tool
def order_event(request: str):
    """处理订单相关请求：查询订单状态、变更订单状态。"""
    result = order_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return f"订单智能体结果：{result['messages'][-1].content}"

@tool
def inventory_event(request: str):
    """处理库存相关请求：查询库存、调整阈值、生成补货建议。"""
    result = inventory_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return f"库存智能体结果：{result['messages'][-1].content}"

@tool
def shipping_event(request: str):
    """处理物流相关请求：跟踪物流轨迹。"""
    result = shipping_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return f"物流智能体结果：{result['messages'][-1].content}"

MAIN_PROMPT = """
你是电商订单处理主管，协调以下三个子智能体，职责分离：
- 订单智能体：order_event(request) —— 查询订单状态、变更订单状态
- 库存智能体：inventory_event(request) —— 查库存、调阈值、补货建议
- 物流智能体：shipping_event(request) —— 跟踪物流

对于复合请求（一句话里包含多个域），请依次调用相关子智能体，
拿到结果后用中文综合回复用户。若请求带条件，请先获取前置信息再决定是否调用后续子智能体。
"""
main_agent = create_agent(
    model=model,
    tools=[order_event, inventory_event, shipping_event],
    system_prompt=MAIN_PROMPT,
    checkpointer=InMemorySaver(),
)
print("主管智能体已创建，可调用子智能体工具：order_event / inventory_event / shipping_event")

主管智能体已创建，可调用子智能体工具：order_event / inventory_event / shipping_event


## 第 5 步：测试单域请求

In [10]:
config = {"configurable": {"thread_id": "user-001"}}

result = main_agent.invoke(
    {"messages": [{"role": "user", "content": "帮我查订单 #ORD-20240115 的状态"}]},
    config=config,
)
print(result["messages"][-1].content)

订单 **#ORD-20240115** 当前状态：**已发货（shipped）**。


## 第 6 步：测试复合跨域请求

一句话包含订单查询 + 条件式物流跟踪 + 库存检查。主管会自动分解并调用对应的子智能体工具。

In [11]:
result = main_agent.invoke(
    {"messages": [{"role": "user", "content": "查订单 #ORD-20240115 的状态，如果已发货就跟踪物流，没货就检查库存并提醒补货"}]},
    config=config,
)
print(result["messages"][-1].content)

已查询到订单 **#ORD-20240115** 状态为 **shipped（已发货）**，因此我已为您跟踪物流：

- **承运商**：顺丰  
- **运单号**：SF1234567890  
- **最新状态**：已到达 **北京市朝阳区转运中心**  
- **当前位置**：北京市朝阳区转运中心  
- **预计到达时间（ETA）**：**2024-01-16**

如您希望我继续盯进度（下一节点/是否签收），告诉我即可。
